In [16]:
import os
import cv2 as cv
import numpy as np
import pandas as pd 
import tensorflow as tf
import matplotlib.pyplot as plt

In [17]:
x=[]
y=[]
base='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
for i in os.listdir(base):
    sec_base=base+'/'+i
    for j in os.listdir(sec_base):
        img=plt.imread(sec_base+'/'+j)
        img=cv.cvtColor(img,cv.COLOR_BGR2RGB)
        img=cv.resize(img,(128,128))
        x.append(img)
        y.append(i)

In [18]:
X_train=np.array(x)
y_train=np.array(y)

In [19]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_train=le.fit_transform(y_train)

In [20]:
model=tf.keras.models.Sequential()

# CNN
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
model.add(base_model)
model.add(tf.keras.layers.GlobalAveragePooling2D())
# ANN

model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(4,activation='softmax'))

In [21]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │        20,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,278,820 (8.69 MB)

 Trainable params: 20,836 (81.39 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [22]:
@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        pred = model(X, training=True)
        loss = loss_fn(y, pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [23]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
dataset = dataset.shuffle(X_train.shape[0]).batch(batch_size).prefetch(tf.data.AUTOTUNE) 

epochs = 10
loss_val=[]
for i in range(epochs):
    for X_batch, y_batch in dataset:
        loss = train_step(X_batch, y_batch)
    loss_val.append(loss)
    print("Epoch:", i+1, "Loss:", loss.numpy())

Epoch: 1 Loss: 0.71496624
Epoch: 2 Loss: 0.4621265
Epoch: 3 Loss: 0.48544636
Epoch: 4 Loss: 0.315463
Epoch: 5 Loss: 0.27247903
Epoch: 6 Loss: 0.24925627
Epoch: 7 Loss: 0.2776872
Epoch: 8 Loss: 0.2262327
Epoch: 9 Loss: 0.2500255
Epoch: 10 Loss: 0.27493763


In [24]:
accuracy=np.mean(np.argmax(model.predict(X_train),axis=1)==y_train)
print("Training Accuracy:", accuracy*100)

175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step
Training Accuracy: 92.08928571428572


In [25]:
x=[]
y=[]
base='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
for i in os.listdir(base):
    sec_base=base+'/'+i
    for j in os.listdir(sec_base):
        img=plt.imread(sec_base+'/'+j)
        img=cv.cvtColor(img,cv.COLOR_BGR2RGB)
        img=cv.resize(img,(128,128))
        x.append(img)
        y.append(i)

In [26]:
X_test=np.array(x)
y_test=np.array(y)

In [27]:
y_test=le.transform(y_test)

In [ ]:
accuracy=np.mean(np.argmax(model.predict(X_test),axis=1)==y_t)
print("Testing Accuracy:", accuracy*100)

50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step
Training Accuracy: 81.125
